# Lab 03: Aggregation-axis
## Статистический анализ продаж с помощью NumPy

Кейс: региональная эффективность интернет-магазина. Продажи в тыс. ден. ед. за 8 недель по 5 регионам.
Циклы, Pandas и ручной перебор не используются — только векторизованный NumPy.

In [1]:
import numpy as np
np.set_printoptions(suppress=True, precision=2, linewidth=140)

sales = np.array([
    [120, 125, 130, 128, 135, 140, 138, 145],
    [ 80,  82,  79,  85,  88,  90,  86,  92],
    [150, 148, 155, 160, 158, 165, 220, 170],
    [ 60,  65,  62,  58,  64,  63,  61,  66],
    [100, 105, 102, 108, 110, 112, 109, 115]])
region_names = np.array(["Bishkek", "Osh", "Karakol", "Tokmok", "Jalal-Abad"])

# проверки входных данных
assert sales.shape == (5, 8) and region_names.shape == (5,)
assert (sales > 0).all()
print("Форма:", sales.shape, "| axis=0 — по регионам (для каждой недели), axis=1 — по неделям (для каждого региона)")

Форма: (5, 8) | axis=0 — по регионам (для каждой недели), axis=1 — по неделям (для каждого региона)


## Задание 1. Сравнение регионов

In [2]:
total = sales.sum(axis=1)
mean_w = sales.mean(axis=1)
median_w = np.median(sales, axis=1)

print(f"{'Регион':<11}{'Сумма':>8}{'Среднее':>10}{'Медиана':>10}")
print("\n".join(f"{n:<11}{t:>8}{m:>10.2f}{d:>10.2f}" for n, t, m, d in zip(region_names, total, mean_w, median_w)))

assert total.sum() == sales.sum()   # проверка: сумма по регионам = сумма всего
print("\nЛидер по суммарным продажам:", region_names[total.argmax()], total.max())

# «типичная» выручка — медиана; разница со средним показывает влияние экстремальных недель
gap = np.abs(mean_w - median_w)
print("|среднее − медиана|:", gap)
print("Регион, где среднее сильнее всего оторвано от медианы:", region_names[gap.argmax()])

Регион        Сумма   Среднее   Медиана
Bishkek        1061    132.62    132.50
Osh             682     85.25     85.50
Karakol        1326    165.75    159.00
Tokmok          499     62.38     62.50
Jalal-Abad      861    107.62    108.50

Лидер по суммарным продажам: Karakol 1326
|среднее − медиана|: [0.12 0.25 6.75 0.12 0.88]
Регион, где среднее сильнее всего оторвано от медианы: Karakol


**Выводы.** Лидер по сумме — **Karakol** (1326 тыс.), далее Bishkek (1061), Jalal-Abad (861), Osh (682), Tokmok (499).
Для «типичной» недельной выручки лучше использовать **медиану**: она не зависит от экстремальных недель. Это видно на Karakol:
среднее 165.75 заметно выше медианы 159 — его тянет вверх всплеск 220 на 7-й неделе. У остальных регионов среднее и медиана почти совпадают
(разница ≤ 1 тыс.), т.е. типичная неделя — ≈132.5 (Bishkek), 85.5 (Osh), 62.5 (Tokmok), 108.5 (Jalal-Abad). Честная оценка типичной
недели Karakol — 159, а не 165.75.

## Задание 2. Анализ разброса

In [3]:
std_s = sales.std(axis=1, ddof=1)          # выборочное СКО (8 недель — это выборка)
q1, q3 = np.percentile(sales, [25, 75], axis=1)
iqr = q3 - q1
cv = std_s / mean_w                        # коэффициент вариации — разброс с учётом масштаба
iqr_rel = iqr / median_w                   # относительный IQR (устойчив к выбросам)

print(f"{'Регион':<11}{'СКО':>7}{'IQR':>7}{'CV, %':>8}{'IQR/мед, %':>12}")
print("\n".join(f"{n:<11}{a:>7.2f}{b:>7.2f}{c*100:>8.2f}{d*100:>12.2f}" for n, a, b, c, d in zip(region_names, std_s, iqr, cv, iqr_rel)))

print("\nОт самого стабильного к самому нестабильному по CV:", region_names[np.argsort(cv)])

Регион         СКО    IQR   CV, %  IQR/мед, %
Bishkek       8.35  11.25    6.29        8.49
Osh           4.68   7.00    5.49        8.19
Karakol      23.09  12.50   13.93        7.86
Tokmok        2.67   3.50    4.28        5.60
Jalal-Abad    5.04   6.25    4.68        5.76

От самого стабильного к самому нестабильному по CV: ['Tokmok' 'Jalal-Abad' 'Osh' 'Bishkek' 'Karakol']


**Выводы.** По абсолютному СКО больше всех разбросан Karakol (23.09), меньше всех Tokmok (2.67) — но это отчасти эффект масштаба: у Karakol и продажи больше.
С поправкой на масштаб (CV = СКО/среднее): Karakol — 13.9 % (самый нестабильный), Bishkek — 6.3 %, Osh — 5.5 %, Jalal-Abad — 4.7 %, Tokmok — 4.3 % (самый стабильный).
Относительный IQR (устойчив к выбросам) у Bishkek, Osh и Karakol одинаков (≈8 %), у Tokmok и Jalal-Abad ниже (≈6 %).
Т.е. высокий CV у Karakol объясняется в основном одной аномальной неделей, а не общей нестабильностью «ядра» данных.

## Задание 3. Поиск выбросов (правило IQR)

In [4]:
lower = (q1 - 1.5 * iqr)[:, None]
upper = (q3 + 1.5 * iqr)[:, None]
outlier_mask = (sales < lower) | (sales > upper)

print("Границы [нижняя, верхняя] по регионам:")
print(np.column_stack([lower.ravel(), upper.ravel()]))
print("Число выбросов по регионам:", dict(zip(region_names.tolist(), outlier_mask.sum(axis=1).tolist())))

r_idx, w_idx = np.nonzero(outlier_mask)
print("\nНайденные выбросы:")
print("\n".join(f"{region_names[r]}, неделя {w+1}: значение {sales[r, w]}, "
                f"медиана региона {median_w[r]:.1f}, отклонение {sales[r, w]-median_w[r]:+.1f} "
                f"({(sales[r, w]/median_w[r]-1)*100:+.1f} %)" for r, w in zip(r_idx, w_idx)))

Границы [нижняя, верхняя] по регионам:
[[110.38 155.38]
 [ 71.    99.  ]
 [135.   185.  ]
 [ 55.5   69.5 ]
 [ 94.88 119.88]]
Число выбросов по регионам: {'Bishkek': 0, 'Osh': 0, 'Karakol': 1, 'Tokmok': 0, 'Jalal-Abad': 0}

Найденные выбросы:
Karakol, неделя 7: значение 220, медиана региона 159.0, отклонение +61.0 (+38.4 %)


**Выводы.** Правило IQR находит один потенциальный выброс: **Karakol, неделя 7 — 220** (верхняя граница 185; +38 % к медиане региона 159).
Следующая неделя (170) возвращается к тренду, т.е. это одиночный всплеск.

*Почему может быть реальным:* промо-акция или распродажа, сезонный пик (праздники), крупный разовый корпоративный заказ, запуск рекламной кампании.
Подтверждение — рост числа заказов и среднего чека, всплеск трафика.

*Почему может быть ошибкой:* опечатка при вводе (220 вместо 120 или 170), двойной учёт заказов / дубль выгрузки, возвраты с неверным знаком,
перенос продаж другой недели или региона. Подтверждение — отсутствие всплеска трафика и числа заказов при высокой выручке.
Выход за границу — повод проверить, а не удалять значение. В остальных регионах выбросов нет.

## Задание 4. Структура выручки

In [5]:
grand_total = total.sum()
assert grand_total != 0                      # защита от деления на ноль
share = total / grand_total * 100
order = np.argsort(-share)

print(f"Общая выручка: {grand_total}")
print("\n".join(f"{region_names[i]:<11}{share[i]:>7.2f} %" for i in order))
print("Сумма долей:", share.sum())
print("Доля топ-1:", share.max().round(2), "% | топ-2:", np.sort(share)[-2:].sum().round(2), "%")

hhi = ((share / 100) ** 2).sum()             # индекс Херфиндаля–Хиршмана
print("HHI:", hhi.round(3), "(равномерное распределение по 5 регионам = 0.2)")

Общая выручка: 4429
Karakol      29.94 %
Bishkek      23.96 %
Jalal-Abad   19.44 %
Osh          15.40 %
Tokmok       11.27 %
Сумма долей: 100.0
Доля топ-1: 29.94 % | топ-2: 53.89 %
HHI: 0.221 (равномерное распределение по 5 регионам = 0.2)


**Выводы.** Karakol даёт **29.9 %** выручки, Bishkek — 24.0 %, Jalal-Abad — 19.4 %, Osh — 15.4 %, Tokmok — 11.3 %. Два крупнейших региона дают ≈54 %.
HHI ≈ 0.21 при минимуме 0.2 для 5 равных регионов — структура близка к сбалансированной. Зависимость от крупнейшего региона **умеренная**:
потеря Karakol убрала бы около 30 % выручки, но ни один регион не доминирует (нет доли > 1/3). Доля Karakol при этом слегка завышена выбросом 220.

## Задание 5. Сравнение периодов

In [6]:
first4, last4 = sales[:, :4], sales[:, 4:]
base = first4.mean(axis=1)
new = last4.mean(axis=1)
abs_change = new - base

# процентное изменение определяем только там, где база != 0, иначе NaN
safe_base = np.where(base == 0, 1, base)
pct_change = np.where(base == 0, np.nan, abs_change / safe_base * 100)

print(f"{'Регион':<11}{'Нед.1-4':>9}{'Нед.5-8':>9}{'Δ абс.':>9}{'Δ, %':>8}")
print("\n".join(f"{n:<11}{b:>9.2f}{c:>9.2f}{d:>9.2f}{p:>8.2f}" for n, b, c, d, p in zip(region_names, base, new, abs_change, pct_change)))

store_growth = (last4.sum() - first4.sum()) / first4.sum() * 100
print(f"\nМагазин в целом: {store_growth:.2f} %")
print("Рост выше среднего по магазину:", region_names[pct_change > store_growth])

Регион       Нед.1-4  Нед.5-8   Δ абс.    Δ, %
Bishkek       125.75   139.50    13.75   10.93
Osh            81.50    89.00     7.50    9.20
Karakol       153.25   178.25    25.00   16.31
Tokmok         61.25    63.50     2.25    3.67
Jalal-Abad    103.75   111.50     7.75    7.47

Магазин в целом: 10.70 %
Рост выше среднего по магазину: ['Bishkek' 'Karakol']


**Выводы.** Все регионы растут. Лучший рост у Karakol (+25 тыс. в неделю, +16.3 %), но он частично обеспечен выбросом (220) во второй половине периода.
Bishkek (+10.9 %) растёт вровень с магазином (+10.7 %). Osh (+9.2 %) и Jalal-Abad (+7.5 %) растут медленнее магазина, а **Tokmok** — всего +3.7 % (+2.25 тыс.),
то есть почти стагнирует. Нулевых базовых значений нет, поэтому процентное изменение определено везде; код на этот случай возвращает NaN.

## Задание 6. Устойчивость результата

In [7]:
clean = np.where(outlier_mask, np.nan, sales)        # выбросы исключаем (NaN), остальные недели не трогаем

total_c = np.nansum(clean, axis=1)
mean_c = np.nanmean(clean, axis=1)
median_c = np.nanmedian(clean, axis=1)
n_c = (~np.isnan(clean)).sum(axis=1)

print(f"{'Регион':<11}{'Сумма до':>9}{'после':>7}{'Ср. до':>9}{'после':>8}{'Мед. до':>9}{'после':>7}{'нед.':>5}")
print("\n".join(f"{n:<11}{a:>9}{b:>7.0f}{c:>9.2f}{d:>8.2f}{e:>9.2f}{f:>7.2f}{k:>5}"
                for n, a, b, c, d, e, f, k in zip(region_names, total, total_c, mean_w, mean_c, median_w, median_c, n_c)))

print("\nРейтинг по сумме до:      ", region_names[np.argsort(-total)])
print("Рейтинг по сумме после:   ", region_names[np.argsort(-total_c)])
print("Рейтинг по среднему до:   ", region_names[np.argsort(-mean_w)])
print("Рейтинг по среднему после:", region_names[np.argsort(-mean_c)])

top2 = np.sort(mean_c)[-2:]
print("Отрыв лидера от 2-го места по среднему после очистки:", top2[1] - top2[0])

Регион      Сумма до  после   Ср. до   после  Мед. до  после нед.
Bishkek         1061   1061   132.62  132.62   132.50 132.50    8
Osh              682    682    85.25   85.25    85.50  85.50    8
Karakol         1326   1106   165.75  158.00   159.00 158.00    7
Tokmok           499    499    62.38   62.38    62.50  62.50    8
Jalal-Abad       861    861   107.62  107.62   108.50 108.50    8

Рейтинг по сумме до:       ['Karakol' 'Bishkek' 'Jalal-Abad' 'Osh' 'Tokmok']
Рейтинг по сумме после:    ['Karakol' 'Bishkek' 'Jalal-Abad' 'Osh' 'Tokmok']
Рейтинг по среднему до:    ['Karakol' 'Bishkek' 'Jalal-Abad' 'Osh' 'Tokmok']
Рейтинг по среднему после: ['Karakol' 'Bishkek' 'Jalal-Abad' 'Osh' 'Tokmok']
Отрыв лидера от 2-го места по среднему после очистки: 25.375


**Выводы.** Исключение 220 меняет только Karakol: сумма 1326 → 1106 (−220, −16.6 %), среднее 165.75 → 158.00, медиана 159 → 158.
Остальные регионы не затронуты. Рейтинги по сумме и по среднему **не изменились**: Karakol остаётся лидером (среднее 158 против 132.6 у Bishkek),
порядок — Karakol, Bishkek, Jalal-Abad, Osh, Tokmok.

Сумма региона после очистки считается по 7 неделям, а не по 8, поэтому суммы корректнее сравнивать осторожно — надёжнее смотреть на среднее.
В общем случае удаление наблюдений **может** изменить рейтинг, если выброс велик относительно разрыва между соседями (например, при близких Bishkek и Karakol).
Здесь отрыв достаточен, поэтому вывод устойчив. Медиана почти не чувствительна к выбросу (159 → 158), среднее — чувствительно.

## Задание 7. Поиск проблемных регионов

In [8]:
# Критерий: регион проблемный, если выполнено >= 2 из 3 условий
#  A) отставание: рост «последние 4 vs первые 4 недели» ниже, чем в целом по магазину
#  B) нестабильность: коэффициент вариации (ddof=1) > 10 %
#  C) слабая недельная динамика: падений «неделя к неделе» >= 4 из 7
# Размер выручки сам по себе НЕ признак (малый регион ≠ проблемный); доля выводится для контекста.

declines = (np.diff(sales, axis=1) < 0).sum(axis=1)
flag_A = pct_change < store_growth
flag_B = cv > 0.10
flag_C = declines >= 4
n_flags = flag_A.astype(int) + flag_B + flag_C
problem = n_flags >= 2

print(f"{'Регион':<11}{'Рост,%':>8}{'CV,%':>7}{'Падений':>9}{'A':>3}{'B':>3}{'C':>3}{'Флагов':>8}{'Доля,%':>8}  Проблемный")
print("\n".join(f"{n:<11}{g:>8.2f}{c*100:>7.2f}{d:>9}{int(a):>3}{int(b):>3}{int(c_):>3}{k:>8}{s:>8.2f}  {'ДА' if p else 'нет'}"
                for n, g, c, d, a, b, c_, k, s, p in zip(region_names, pct_change, cv, declines, flag_A, flag_B, flag_C, n_flags, share, problem)))

# проверка чувствительности: CV без выбросов
cv_clean = np.nanstd(clean, axis=1, ddof=1) / mean_c
print("\nCV после исключения выбросов, %:", (cv_clean * 100).round(2))

Регион       Рост,%   CV,%  Падений  A  B  C  Флагов  Доля,%  Проблемный
Bishkek       10.93   6.29        2  0  0  0       0   23.96  нет
Osh            9.20   5.49        2  1  0  0       1   15.40  нет
Karakol       16.31  13.93        3  0  1  0       1   29.94  нет
Tokmok         3.67   4.28        4  1  0  1       2   11.27  ДА
Jalal-Abad     7.47   4.68        2  1  0  0       1   19.44  нет

CV после исключения выбросов, %: [6.29 5.49 4.97 4.28 4.68]


**Обоснование критерия.** Он объединяет три независимых признака: динамику (A), разброс (B) и устойчивость недельного тренда (C). Требование ≥ 2 флагов защищает
от ложных тревог из-за одного показателя. Объём выручки намеренно не включён: Tokmok — маленький рынок, но размер сам по себе не проблема. Рост сравнивается со средним по магазину,
т.е. с поправкой на масштаб.

**Результат.** Проблемным оказывается **Tokmok** (флаги A и C: рост лишь 3.7 % при +10.7 % по магазину, 4 из 7 недельных переходов — спад; продажи при этом очень стабильны, CV 4.3 %).
Это «стабильная стагнация»: доля всего 11.3 %, масштаб потерь для магазина невелик, но тренд самый слабый.
**Karakol** получает только флаг B (CV 13.9 %), и он целиком объясняется выбросом: без него CV ≈ 4–5 %. Регион не проблемный, а «требует проверки данных».
Osh и Jalal-Abad отстают по росту (только флаг A) — под наблюдением. Bishkek (рост ≈ магазин) — в норме.

## Задание 8. Аналитическая записка для руководства

**1. Какие регионы формируют основную выручку.** Karakol (29.9 %) и Bishkek (24.0 %) — вместе ≈54 %; с Jalal-Abad (19.4 %) — около 73 %.
Зависимость от одного региона умеренная (нет доли > 1/3), но доля Karakol завышена одной аномальной неделей.

**2. Где наблюдаются нестабильные продажи.** По CV — только Karakol (13.9 %), и это следствие одного всплеска (неделя 7: 220). Без него регион стабилен.
Остальные регионы стабильны (CV 4–6 %). Отдельно Tokmok: нестабильности нет, но рост почти нулевой, а в большинстве недельных переходов — спад.

**3. Какие отклонения требуют проверки.** (а) Karakol, неделя 7 = 220 (+38 % к медиане) — корректность записи, дубли, наличие акции;
(б) Tokmok — причины слабой динамики (+3.7 % против +10.7 % по магазину); (в) Osh и Jalal-Abad — рост ниже среднего по магазину.

**4. Какие дополнительные данные нужны.** Число заказов и средний чек по неделям; трафик и конверсия; маркетинговые акции и расходы по регионам и неделям;
сезонность и более длинная история (8 недель мало для выводов о тренде); численность населения / ёмкость рынка региона;
цены, ассортимент, остатки и логистика; возвраты и отмены; маржинальность; данные по конкурентам.

*Ограничение:* 8 наблюдений на регион — выводы предварительные; выбросы и тренды нужно подтверждать на более длинной истории.

## Дополнительное задание. Ранжирование по разным метрикам

In [9]:
rank_total = np.argsort(np.argsort(-total)) + 1          # 1 = лучший
rank_mean = np.argsort(np.argsort(-mean_w)) + 1
rank_stab = np.argsort(np.argsort(cv)) + 1               # меньше CV = стабильнее
rank_growth = np.argsort(np.argsort(-pct_change)) + 1

print(f"{'Регион':<11}{'Сумма':>7}{'Среднее':>9}{'Стабильн.':>11}{'Рост':>6}")
print("\n".join(f"{n:<11}{a:>7}{b:>9}{c:>11}{d:>6}" for n, a, b, c, d in zip(region_names, rank_total, rank_mean, rank_stab, rank_growth)))
print("\nРанги по сумме и среднему совпадают:", np.array_equal(rank_total, rank_mean))
print("Лидер по сумме:", region_names[rank_total == 1], "| лидер по стабильности:", region_names[rank_stab == 1])

Регион       Сумма  Среднее  Стабильн.  Рост
Bishkek          2        2          4     2
Osh              4        4          3     3
Karakol          1        1          5     1
Tokmok           5        5          1     5
Jalal-Abad       3        3          2     4

Ранги по сумме и среднему совпадают: True
Лидер по сумме: ['Karakol'] | лидер по стабильности: ['Tokmok']


**Выводы.** Ранги по общей и средней выручке совпадают (у всех регионов по 8 недель, среднее = сумма / 8, поэтому порядок один).
По стабильности (CV) порядок другой: Tokmok (1) — Jalal-Abad (2) — Osh (3) — Bishkek (4) — Karakol (5). Лидер по объёму Karakol — **последний** по стабильности,
а самый маленький Tokmok — первый. Объём и стабильность — разные свойства: объём зависит от размера рынка, CV — от относительных колебаний, а высокая выручка
может обеспечиваться отдельными всплесками. Эффективность тоже нельзя мерить одним объёмом: по росту лидирует Karakol, а самый стабильный Tokmok растёт хуже всех.
Для управленческих решений нужен набор метрик (объём, типичное значение, разброс, динамика), а не один рейтинг.